# Audio-Conditioned Whisper→SONAR Alignment: Fitting `W` on Real LibriSpeech Audio

**What changes versus `whisper_sonar_alignment_experiment.ipynb` (Exp-01).** Exp-01 validated the
linear map using **silence-conditioned** decoder states (text teacher-forced over a silent
spectrogram) and WikiText sentences — a deliberate worst case, and the only option without parallel
audio. But at deployment the decoder's states are conditioned on **real audio** through
cross-attention; the biasing gate will consume *those* states, not silence states. LibriSpeech gives
us the parallel (audio, reference text) pairs to close that gap:

```
LibriSpeech audio ──► Whisper encoder ──► decoder (teacher-forced on the reference) ──► pooled states
LibriSpeech text  ──────────────────────────────────────────────────────────────────► SONAR embedding
                                             fit  W_audio : states → embeddings
```

**Three questions this notebook answers:**

1. **H-audio:** does the linear map hold on the deployment distribution — audio-conditioned states →
   SONAR, evaluated by held-out retrieval on dev-clean and dev-other separately? (dev-other also
   tests robustness to harder acoustics.)
2. **H-transfer:** how much does the existing silence-fitted `whisper_to_sonar_W.pt` lose when
   evaluated on audio-conditioned states? If the drop is small, Exp-01's artifact is deployable as
   is; if large, `W_audio` (produced here) replaces it. This directly informs **spec §10.3**
   (calibration-corpus decision: WikiText/silence vs domain-matched LibriSpeech).
3. **Layer choice, revisited:** the best decoder layer may differ under audio conditioning — the
   full layer sweep runs again, including the prefix probe (the discriminative test; full-sentence
   retrieval saturates).

**Policy note.** Phase 0 supporting analysis (like the template diagnostic): no decoding, no tuning
of experimental conditions — it informs a spec decision point. Data discipline: **TUNE shards only**
(the same `sorted → Random(42).shuffle → 25%` rule as Phase 1), DEVTEST and test splits untouched.

**Runtime.** Extraction is one teacher-forced forward per batch (no beam search): ~2×1000 utterances
≈ 15–30 min on GPU plus SONAR embedding of the references. `FAST` (auto on CPU) shrinks to 150 per
split.


## Section 0.1 — Dependencies (same stack as Exp-01 + audio decoding)

In [ ]:
# Run once, then restart the kernel.
# %pip install -U torch transformers datasets scikit-learn scipy pandas matplotlib soundfile librosa
# %pip install sonar-space


## Section 0.2 — Imports, seed, device

In [ ]:
import os, math, random, re, json, itertools
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)


## Section 0.3 — Configuration

- `n_per_split` — utterances drawn from each split's TUNE shard (1,000 each ≈ 2,000 fitting pairs;
  Exp-01 used 2,000, so scale is comparable).
- `w_silence_path` — Exp-01's artifact for the transfer comparison (skipped gracefully if absent on
  this machine).
- `text_norm_for_states` — **both** the teacher-forced decoder input and the SONAR input use the
  project's shared normalization (lowercase, `[a-z' ]`). LibriSpeech references are ALL-CAPS; feeding
  them raw would give Whisper's BPE an out-of-distribution casing pattern. Lowercasing is the
  neutral choice; the residual gap to deployment (Whisper's own cased/punctuated output) is noted in
  the rubric.
- Grids and probe settings mirror Exp-01 for comparability.


In [ ]:
FAST = (DEVICE == "cpu")

CONFIG = {
    "whisper_model": "openai/whisper-base",
    "splits": {"dev-clean": ("clean", "validation"), "dev-other": ("other", "validation")},
    "tune_frac": 0.25,
    "n_per_split": 150 if FAST else 1000,
    "max_audio_s": 29.0,
    "batch_size": 8,
    "test_frac": 0.10,
    "val_frac": 0.10,
    "ridge_alphas": [1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0],
    "prefix_fracs": [0.25, 0.50, 0.75, 1.00],
    "prefix_probe_n": 100 if FAST else 150,
    "w_silence_path": "whisper_to_sonar_W.pt",
}
print("FAST:", FAST)
CONFIG


## Section 1 — LibriSpeech TUNE-shard utterances (audio + reference)

Line-by-line:
1. For each split: stream, collect the **full ID list first** (text-only pass, audio column removed —
   cheap), apply the frozen shard rule, keep the TUNE 25%. The printed shard SHA must match Phase 1's
   for the same split.
2. Second pass streams audio for TUNE members only, up to `n_per_split`, skipping >29 s utterances
   (counted). Selection under the cap is deterministic: TUNE IDs sorted, seeded shuffle, take in
   order as the stream yields them from the resulting membership set — the recorded ID list in the
   artifacts is what the reviewer reproduces from.
3. Each kept item stores the raw audio array, the raw reference, and `text_norm` (shared
   normalization) — the string both models consume.


In [ ]:
from datasets import load_dataset, Audio
import hashlib

def norm(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def sha(obj):
    return hashlib.sha256(json.dumps(sorted(obj)).encode()).hexdigest()[:16]

def shard_ids(ids, tune_frac, seed):
    ordered = sorted(ids)
    random.Random(seed).shuffle(ordered)
    k = int(len(ordered) * tune_frac)
    return set(ordered[:k]), set(ordered[k:])

def get_audio(sample):
    a = sample["audio"]
    if isinstance(a, dict) and a.get("array") is not None:
        return np.asarray(a["array"], dtype=np.float32)
    if hasattr(a, "get_all_samples"):
        return a.get_all_samples().data.numpy().astype(np.float32).flatten()
    import soundfile as sf
    return sf.read(a["path"], dtype="float32")[0]

DATA = {}
for split_name, (hf_config, hf_split) in CONFIG["splits"].items():
    ids_stream = load_dataset("openslr/librispeech_asr", hf_config,
                              split=hf_split, streaming=True).remove_columns(["audio"])
    all_ids = [s["id"] for s in ids_stream]
    tune, _ = shard_ids(all_ids, CONFIG["tune_frac"], SEED)
    print(f"{split_name}: {len(all_ids)} utts, TUNE {len(tune)} (sha {sha(tune)})")

    stream = load_dataset("openslr/librispeech_asr", hf_config,
                          split=hf_split, streaming=True)
    stream = stream.cast_column("audio", Audio(sampling_rate=16000))
    items, skipped = [], 0
    for s in stream:
        if len(items) >= CONFIG["n_per_split"]:
            break
        if s["id"] not in tune:
            continue
        audio = get_audio(s)
        if len(audio) > 16000 * CONFIG["max_audio_s"]:
            skipped += 1
            continue
        items.append({"id": s["id"], "audio": audio,
                      "text": s["text"], "text_norm": norm(s["text"])})
    DATA[split_name] = items
    print(f"  kept {len(items)} TUNE utterances ({skipped} skipped >29 s)")


## Section 2 — Audio-conditioned decoder states (every layer)

The extraction mirrors Exp-01 **except the conditioning**: the encoder consumes each utterance's real
log-mel features (no caching possible — every utterance differs), and the decoder is teacher-forced
on that utterance's normalized reference. Pooling is byte-identical to Exp-01 and to the runtime
gate: mask out the 4 prompt specials and the trailing `<|endoftext|>`, mean-pool the text-token
positions, L2-normalize — per layer. Output per split: `[n_layers+1, N, 512]`.

Why teacher-forcing (not decoding): we need states for the *reference* text under this audio — the
exact pairing SONAR embeds. Decoding would introduce Whisper's errors into the text side and break
the parallel-pair construction; the deployment gap that remains (Whisper's own casing/punctuation) is
a second-order effect noted in the rubric.


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(CONFIG["whisper_model"])
whisper = (WhisperForConditionalGeneration
           .from_pretrained(CONFIG["whisper_model"]).to(DEVICE).eval())
tok = processor.tokenizer
tok.set_prefix_tokens(language="english", task="transcribe")
N_PREFIX = len(tok.prefix_tokens)

@torch.no_grad()
def audio_text_states(items, batch_size):
    chunks = []
    for i in range(0, len(items), batch_size):
        b = items[i:i + batch_size]
        feats = processor.feature_extractor(
            [x["audio"] for x in b], sampling_rate=16000,
            return_tensors="pt").input_features.to(DEVICE)
        enc = tok([x["text_norm"] for x in b], return_tensors="pt", padding=True)
        ids, attn = enc.input_ids.to(DEVICE), enc.attention_mask.to(DEVICE)
        out = whisper(input_features=feats, decoder_input_ids=ids,
                      output_hidden_states=True, return_dict=True)
        hs = torch.stack(out.decoder_hidden_states)          # [L+1, B, T, D]
        mask = attn.clone()
        mask[:, :N_PREFIX] = 0
        mask.scatter_(1, attn.sum(1, keepdim=True) - 1, 0)
        mask = mask.unsqueeze(0).unsqueeze(-1).float()
        pooled = (hs * mask).sum(2) / mask.sum(2).clamp(min=1.0)
        chunks.append(F.normalize(pooled, dim=-1).float().cpu())
        if (i // batch_size) % 10 == 0:
            print(f"  batch {i // batch_size + 1}/{math.ceil(len(items) / batch_size)}")
    return torch.cat(chunks, dim=1)                          # [L+1, N, D]

STATES = {}
for split_name, items in DATA.items():
    print(f"extracting {split_name} ({len(items)} utts)...")
    STATES[split_name] = audio_text_states(items, CONFIG["batch_size"])
N_LAYERS = next(iter(STATES.values())).shape[0]
print({s: tuple(v.shape) for s, v in STATES.items()})


## Section 3 — SONAR embeddings of the same references

One embedding per utterance reference (normalized text — identical string to the teacher-forced
input), unit-norm. CPU pipeline for fairseq2 compatibility; one-off cost.


In [ ]:
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

t2vec = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder",
                                     tokenizer="text_sonar_basic_encoder",
                                     device=torch.device("cpu"))

Y = {}
for split_name, items in DATA.items():
    with torch.no_grad():
        e = t2vec.predict([x["text_norm"] for x in items],
                          source_lang="eng_Latn", batch_size=64).float()
    Y[split_name] = F.normalize(e, dim=-1).cpu()
    print(split_name, tuple(Y[split_name].shape))


## Section 4 — Train / val / test partition

Per split: seeded shuffle of utterance indices → 80% train / 10% val (Ridge α selection) / 10% test
(all verdict metrics). `W_audio` is fitted on the **concatenated train sets of both splits** (one
domain-matched matrix, as the spec's calibration artifact would be), but evaluated on each split's
test pool **separately** — dev-other's pool answers the robustness question.


In [ ]:
PART = {}
for split_name, items in DATA.items():
    idx = list(range(len(items)))
    random.Random(SEED).shuffle(idx)
    n = len(idx)
    n_test, n_val = int(n * CONFIG["test_frac"]), int(n * CONFIG["val_frac"])
    PART[split_name] = {"test": idx[:n_test],
                        "val": idx[n_test:n_test + n_val],
                        "train": idx[n_test + n_val:]}
    print(split_name, {k: len(v) for k, v in PART[split_name].items()},
          f"chance top-1 = {100.0 / max(n_test, 1):.2f}%")


## Section 5 — Fitting and evaluation machinery (Exp-01's, verbatim)

`fit_ols` (exact least squares), `fit_ridge` (α by validation top-1), `evaluate_map` (project →
normalize → full cosine matrix → top-k retrieval, paired vs random cosine, mean rank). Identical
math to Exp-01 so numbers are directly comparable across the two experiments.


In [ ]:
def fit_ols(X, Yt):
    Wm, *_ = np.linalg.lstsq(X, Yt, rcond=None)
    return Wm

def evaluate_map(Wm, X, Yt):
    P = F.normalize(torch.from_numpy(X).float() @ torch.from_numpy(Wm).float(), dim=-1)
    T = F.normalize(torch.from_numpy(Yt).float(), dim=-1)
    sim = P @ T.T
    d = sim.diag()
    n = sim.shape[0]
    rank = (sim > d.unsqueeze(1)).sum(dim=1)
    off = (sim.sum() - d.sum()) / (n * n - n)
    return {"top1": (rank < 1).float().mean().item(),
            "top5": (rank < 5).float().mean().item(),
            "top10": (rank < 10).float().mean().item(),
            "mean_rank": rank.float().mean().item() + 1.0,
            "paired_cos": d.mean().item(),
            "random_cos": off.item()}

def fit_ridge(Xtr, Ytr, Xval, Yval, alphas):
    from sklearn.linear_model import Ridge
    best = (None, None, -1.0)
    for a in alphas:
        reg = Ridge(alpha=a, fit_intercept=False).fit(Xtr, Ytr)
        Wm = reg.coef_.T
        v = evaluate_map(Wm, Xval, Yval)["top1"]
        if v > best[2]:
            best = (Wm, a, v)
    return best

print("machinery ready")


## Section 6 — Layer sweep: fit `W_audio`, evaluate per split

For each layer: assemble train/val matrices by concatenating both splits' partitions; fit OLS and
Ridge; evaluate on **each split's own test pool**. The table's `top1` per split is the H-audio
verdict; the layer ranking (compare with Exp-01's layer 4) shows whether audio conditioning moves
the sweet spot.


In [ ]:
rows, fitted = [], {}
for layer in range(N_LAYERS):
    Xtr = np.concatenate([STATES[s][layer].numpy()[PART[s]["train"]] for s in DATA])
    Ytr = np.concatenate([Y[s].numpy()[PART[s]["train"]] for s in DATA])
    Xval = np.concatenate([STATES[s][layer].numpy()[PART[s]["val"]] for s in DATA])
    Yval = np.concatenate([Y[s].numpy()[PART[s]["val"]] for s in DATA])

    W_ols = fit_ols(Xtr, Ytr)
    W_rdg, alpha, _ = fit_ridge(Xtr, Ytr, Xval, Yval, CONFIG["ridge_alphas"])
    fitted[layer] = {"ols": W_ols, "ridge": W_rdg, "alpha": alpha}

    for s in DATA:
        Xte = STATES[s][layer].numpy()[PART[s]["test"]]
        Yte = Y[s].numpy()[PART[s]["test"]]
        for mname, Wm in [("OLS", W_ols), (f"Ridge(α={alpha:g})", W_rdg)]:
            rows.append({"layer": layer, "method": mname, "split": s,
                         **{k: round(v, 4) for k, v in evaluate_map(Wm, Xte, Yte).items()}})
    done = [r for r in rows if r["layer"] == layer and r["method"].startswith("Ridge")]
    print(f"layer {layer}: " + "  ".join(f"{r['split']} top1={r['top1']:.1%}" for r in done))

results = pd.DataFrame(rows)
summary = (results.groupby(["layer", "method"])["top1"].mean()
           .reset_index().sort_values("top1", ascending=False))
summary.head(8)


## Section 7 — Best configuration + shuffled-pairs control

Best (layer, method) by mean test top-1 across splits; the shuffled control refits Ridge at that
layer on permuted pairs — its score is the "regression found nothing" floor that real numbers must
dwarf, exactly as in Exp-01.


In [ ]:
best_row = summary.iloc[0]
BEST_LAYER = int(best_row["layer"])
BEST_METHOD = best_row["method"]
W_AUDIO = fitted[BEST_LAYER]["ridge" if BEST_METHOD.startswith("Ridge") else "ols"]
print(f"best: layer {BEST_LAYER}, {BEST_METHOD}, mean test top1 = {best_row['top1']:.1%}")

Xtr = np.concatenate([STATES[s][BEST_LAYER].numpy()[PART[s]["train"]] for s in DATA])
Ytr = np.concatenate([Y[s].numpy()[PART[s]["train"]] for s in DATA])
Xval = np.concatenate([STATES[s][BEST_LAYER].numpy()[PART[s]["val"]] for s in DATA])
Yval = np.concatenate([Y[s].numpy()[PART[s]["val"]] for s in DATA])
perm = np.random.RandomState(SEED).permutation(len(Ytr))
W_ctrl, _, _ = fit_ridge(Xtr, Ytr[perm], Xval, Yval, CONFIG["ridge_alphas"])
for s in DATA:
    Xte = STATES[s][BEST_LAYER].numpy()[PART[s]["test"]]
    Yte = Y[s].numpy()[PART[s]["test"]]
    m = evaluate_map(W_ctrl, Xte, Yte)
    print(f"shuffled control on {s}: top1={m['top1']:.2%} (chance "
          f"{1.0 / len(PART[s]['test']):.2%})")


## Section 8 — H-transfer: the silence-fitted `W` on audio states

Load Exp-01's `whisper_to_sonar_W.pt` (silence/WikiText-calibrated, layer 4) and evaluate it on the
**audio-conditioned test states** — layer 4 states for a like-for-like comparison, plus this run's
best layer. The gap between `W_audio` and `W_silence` on the same pools is the spec §10.3 answer:

- **small gap** → silence calibration transfers; either artifact works, WikiText's easy data
  pipeline stays acceptable;
- **large gap** → deployment needs audio-conditioned calibration; `W_audio` (saved below) becomes
  the project artifact and the spec's §4 is updated accordingly.


In [ ]:
transfer_rows = []
if os.path.exists(CONFIG["w_silence_path"]):
    W_sil = torch.load(CONFIG["w_silence_path"], map_location="cpu").float().numpy()
    for s in DATA:
        Yte = Y[s].numpy()[PART[s]["test"]]
        for layer, label in [(4, "layer4 (Exp-01 choice)"), (BEST_LAYER, f"layer{BEST_LAYER} (this run)")]:
            Xte = STATES[s][layer].numpy()[PART[s]["test"]]
            m_sil = evaluate_map(W_sil, Xte, Yte)
            transfer_rows.append({"split": s, "eval_layer": label,
                                  "matrix": "W_silence(Exp-01)", **{k: round(v, 4) for k, v in m_sil.items()}})
        Xte4 = STATES[s][BEST_LAYER].numpy()[PART[s]["test"]]
        m_aud = evaluate_map(W_AUDIO, Xte4, Yte)
        transfer_rows.append({"split": s, "eval_layer": f"layer{BEST_LAYER} (this run)",
                              "matrix": "W_audio(this nb)", **{k: round(v, 4) for k, v in m_aud.items()}})
    transfer_df = pd.DataFrame(transfer_rows)
    display(transfer_df)
else:
    print(f"{CONFIG['w_silence_path']} not found on this machine — transfer comparison skipped.")
    transfer_df = pd.DataFrame()


## Section 9 — Prefix probe under audio conditioning

The runtime-relevant test (full-sentence retrieval saturates): pool only the first 25/50/75/100% of
text-token states — **under real audio** — project with `W_audio`, retrieve against each split's full
test pool. Causality makes prefix states inside one teacher-forced pass identical to running the
prefix alone, so one forward per utterance suffices. Compare the curve with Exp-01's
(0.29 / 0.77 / 0.975 / 1.00): audio conditioning gives the decoder acoustic evidence about the
*whole* utterance from step one via cross-attention, so early-prefix retrieval may improve — that
would directly benefit the [[semantic-gating]] use case where early-decode estimates were the weak
zone.


In [ ]:
@torch.no_grad()
def token_states_one(item, layer):
    feats = processor.feature_extractor(item["audio"], sampling_rate=16000,
                                        return_tensors="pt").input_features.to(DEVICE)
    ids = tok(item["text_norm"], return_tensors="pt").input_ids.to(DEVICE)
    out = whisper(input_features=feats, decoder_input_ids=ids,
                  output_hidden_states=True, return_dict=True)
    return out.decoder_hidden_states[layer][0][N_PREFIX:-1].float().cpu()

W_t = torch.from_numpy(W_AUDIO).float()
probe_rows = []
for s in DATA:
    test_idx = PART[s]["test"][:CONFIG["prefix_probe_n"]]
    pool = F.normalize(Y[s][PART[s]["test"]].float(), dim=-1)
    own = {orig: k for k, orig in enumerate(PART[s]["test"])}
    token_cache = [(own[i], token_states_one(DATA[s][i], BEST_LAYER)) for i in test_idx]
    for frac in CONFIG["prefix_fracs"]:
        preds, owns = [], []
        for pos, ts in token_cache:
            k = max(1, math.ceil(frac * ts.shape[0]))
            preds.append(F.normalize(ts[:k].mean(0, keepdim=True), dim=-1))
            owns.append(pos)
        P = F.normalize(torch.cat(preds) @ W_t, dim=-1)
        sim = P @ pool.T
        d = sim[torch.arange(len(owns)), torch.tensor(owns)]
        rank = (sim > d.unsqueeze(1)).sum(1)
        probe_rows.append({"split": s, "prefix": f"{int(frac*100)}%",
                           "top1": round((rank < 1).float().mean().item(), 4),
                           "top5": round((rank < 5).float().mean().item(), 4),
                           "paired_cos": round(d.mean().item(), 4)})
prefix_df = pd.DataFrame(probe_rows)
prefix_df.pivot(index="prefix", columns="split", values="top1")


## Section 10 — Visual summary

Layer sweep per split (Ridge), prefix curves vs Exp-01's silence-conditioned curve, and — when the
Exp-01 artifact is present — the transfer comparison bars.


In [ ]:
fig, axes = plt.subplots(1, 3 if len(transfer_df) else 2, figsize=(16, 4.2))

for s in DATA:
    g = results[(results["split"] == s) & (results["method"].str.startswith("Ridge"))] \
        .sort_values("layer")
    axes[0].plot(g["layer"], g["top1"] * 100, marker="o", label=s)
axes[0].set_xlabel("decoder layer"); axes[0].set_ylabel("test top-1 (%)")
axes[0].set_title("W_audio quality by layer (Ridge)"); axes[0].legend()

EXP01_PREFIX = {"25%": 0.29, "50%": 0.765, "75%": 0.975, "100%": 1.00}
for s in DATA:
    g = prefix_df[prefix_df["split"] == s]
    axes[1].plot(g["prefix"], g["top1"], marker="o", label=f"{s} (audio)")
axes[1].plot(list(EXP01_PREFIX.keys()), list(EXP01_PREFIX.values()),
             "k--", marker="x", label="Exp-01 (silence)")
axes[1].set_xlabel("prefix seen"); axes[1].set_ylabel("top-1")
axes[1].set_title("Prefix probe: audio vs silence conditioning"); axes[1].legend()

if len(transfer_df):
    piv = transfer_df[transfer_df["eval_layer"].str.contains("this run")] \
        .pivot(index="split", columns="matrix", values="top1")
    piv.plot.bar(ax=axes[2], rot=0)
    axes[2].set_ylabel("test top-1")
    axes[2].set_title("Transfer: silence-fitted vs audio-fitted W")
plt.tight_layout(); plt.show()


## Section 11 — Artifacts

`whisper_to_sonar_W_audio.pt` — the audio-conditioned matrix at the winning layer — plus the sweep
table, transfer table, prefix table, and a self-describing meta JSON (including the analyzed
utterance IDs for reproducibility).


In [ ]:
torch.save(torch.from_numpy(W_AUDIO).float(), "whisper_to_sonar_W_audio.pt")
results.to_csv("audio_alignment_results.csv", index=False)
prefix_df.to_csv("audio_alignment_prefix.csv", index=False)
if len(transfer_df):
    transfer_df.to_csv("audio_alignment_transfer.csv", index=False)
meta = {
    "config": {k: v for k, v in CONFIG.items() if k != "splits"},
    "best_layer": BEST_LAYER, "best_method": str(BEST_METHOD),
    "mean_test_top1": float(best_row["top1"]),
    "utterance_ids": {s: [DATA[s][i]["id"] for i in range(len(DATA[s]))] for s in DATA},
    "partitions": {s: {k: [DATA[s][i]["id"] for i in v] for k, v in PART[s].items()}
                   for s in DATA},
}
with open("audio_alignment_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved: whisper_to_sonar_W_audio.pt, audio_alignment_{results,prefix,transfer}.csv, "
      "audio_alignment_meta.json")


## Section 12 — Reading the results

**H-audio (does the map hold on deployment-distribution states?)**
- Test top-1 ≫ chance with shuffled control ≈ chance on **both** splits → yes; the linear bridge
  survives real acoustics. Expect dev-other slightly below dev-clean; a *large* dev-other drop means
  acoustic noise leaks into the pooled text states — worth knowing before gating on them.
- Near-ceiling everywhere → use the prefix probe and the transfer table for decisions; full-sentence
  retrieval has no discriminative power at these pool sizes (Exp-01 lesson).

**H-transfer (spec §10.3 decision):**
- `W_silence` within a few points of `W_audio` on audio states → calibration corpus barely matters;
  keep whichever is operationally simpler and record the numbers in the spec.
- `W_audio` clearly ahead → domain/conditioning-matched calibration wins; propose `W_audio` as the
  project artifact (spec §4 update, lead approval required — it changes a frozen input).

**Prefix probe vs Exp-01:** if early-prefix retrieval improves materially over silence conditioning
(0.29 at 25%), the semantic gate's weakest regime — early in decoding, where biasing decisions
actually fire — is stronger than previously believed; this feeds directly into H1a expectations.

**Caveats (by design):** teacher-forced reference text (normalized lowercase) is still one step from
Whisper's own cased/punctuated hypotheses at runtime; pool sizes of ~100–200 make retrieval easier
than Exp-01's 400-pool (compare like with like — chance levels are printed); TUNE-shard data only,
so these pairs overlap the shard later used for (δ, λ) tuning but never the DEVTEST/report shards.

**Wiki follow-up:** after a real run, file the outcome as `wiki/experiments/exp-04-audio-alignment.md`,
update `whisper-sonar-linear-map.md` (calibration recipe + open question §10.3) and `overview.md`,
and append a log entry — per the CLAUDE.md schema.
